# Option 3: Difference-in-Differences
**Replication**: Callaway & Sant'Anna (2021) - Difference-in-Differences with Multiple Time Periods
**Data**: Minimum Wage and Employment (Card & Krueger, 1994; Dube et al., 2010)

**Key Question**: Do minimum wage increases reduce employment?

## 1. Setup and Data Download

In [ ]:
# Install required packages
!pip install -q pandas numpy matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Download minimum wage data
# Using Card & Krueger (1994) data on fast-food employment in NJ and PA
url = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/Ecdat/MinimumWage.csv"

# Alternative: Create simulated staggered DiD data similar to Dube et al.
np.random.seed(42)

# Simulate state-level panel data with staggered treatment
n_states = 50
n_periods = 20
states = [f'State_{i:02d}' for i in range(n_states)]
periods = list(range(2000, 2000 + n_periods))

# Treatment timing: staggered adoption between 2005-2012
treat_time = np.random.choice([2005, 2006, 2007, 2008, 2009, 2010, np.nan], 
                              size=n_states, p=[0.15, 0.15, 0.15, 0.15, 0.15, 0.15, 0.1])

data = []
for i, state in enumerate(states):
    for t in periods:
        treated = 0 if np.isnan(treat_time[i]) else (t >= treat_time[i])
        # Base employment with trend
        base = 100 + 2*(t - 2000) + np.random.normal(0, 5)
        # Treatment effect (positive in this simulation)
        effect = 5 * treated  # Minimum wage increases employment slightly
        # State fixed effect
        state_fe = np.random.normal(0, 10)
        employment = base + effect + state_fe
        
        data.append({
            'state': state,
            'year': t,
            'employment': employment,
            'treated': int(treated),
            'treat_time': treat_time[i]
        })

df = pd.DataFrame(data)

print(f"Dataset shape: {df.shape}")
print(f"\nStates: {df["state"].nunique()}")
print(f"Years: {df["year"].min()} - {df["year"].max()}")
print(f"\nTreatment adoption by year:")
print(df[df["treat_time"].notna()]["treat_time"].value_counts().sort_index())
df.head()

## 2. Data Exploration

In [ ]:
# Visualize treatment adoption
treat_counts = df[df["treat_time"].notna()].groupby("treat_time")["state"].nunique()

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
treat_counts.plot(kind='bar')
plt.xlabel('Treatment Year')
plt.ylabel('Number of States')
plt.title('Staggered Treatment Adoption')
plt.xticks(rotation=45)
plt.grid(alpha=0.3)

# Plot employment trends by treatment status
plt.subplot(1, 2, 2)
treated_states = df[df["treat_time"].notna()]["state"].unique()
control_states = df[df["treat_time"].isna()]["state"].unique()

trend_treated = df[df["state"].isin(treated_states)].groupby("year")["employment"].mean()
trend_control = df[df["state"].isin(control_states)].groupby("year")["employment"].mean()

plt.plot(trend_treated.index, trend_treated.values, label='Treated States', marker='o')
plt.plot(trend_control.index, trend_control.values, label='Control States', marker='s')
plt.xlabel('Year')
plt.ylabel('Employment')
plt.title('Employment Trends')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 3. DiD Implementation

In [ ]:
def compute_att_gt(df, group_col='state', time_col='year', outcome='employment', treatment='treated'):
    """
    Compute ATT(g,t) following Callaway & Sant'Anna (2021)
    
    Group g: units first treated at time g
    Compare to never-treated or not-yet-treated units
    
    ATT(g,t) = [E[Y_t|g] - E[Y_g|g]] - [E[Y_t|control] - E[Y_g|control]]
    
    Returns: DataFrame with ATT(g,t) estimates
    """
    results = []
    
    # Get unique groups and time periods
    groups = sorted(df[df["treat_time"].notna()]["treat_time"].unique())
    times = sorted(df[time_col].unique())
    
    for g in groups:
        # Units first treated at time g
        treated_units = df[df["treat_time"] == g]
        
        # Control: never-treated units
        control_units = df[df["treat_time"].isna()]
        
        for t in times:
            if t < g:
                continue  # Pre-treatment
            
            # DID: (Y_t - Y_g) for treated vs control
            y_t_treat = treated_units[treated_units[time_col] == t][outcome].mean()
            y_g_treat = treated_units[treated_units[time_col] == g][outcome].mean()
            y_t_ctrl = control_units[control_units[time_col] == t][outcome].mean()
            y_g_ctrl = control_units[control_units[time_col] == g][outcome].mean()
            
            if pd.notna(y_t_treat) and pd.notna(y_g_treat) and pd.notna(y_t_ctrl) and pd.notna(y_g_ctrl):
                att_gt = (y_t_treat - y_g_treat) - (y_t_ctrl - y_g_ctrl)
                results.append({'group': g, 'time': t, 'att_gt': att_gt})
    
    return pd.DataFrame(results)

def event_study_plot(att_gt_df):
    """
    Plot dynamic treatment effects by event time
    """
    # Calculate event time (time - group)
    att_gt_df = att_gt_df.copy()
    att_gt_df['event_time'] = att_gt_df['time'] - att_gt_df['group']
    
    # Aggregate by event time
    event_study = att_gt_df.groupby('event_time')['att_gt'].agg(['mean', 'std', 'count'])
    event_study = event_study[event_study['count'] >= 3]  # Require at least 3 groups
    event_study['ci_lower'] = event_study['mean'] - 1.96 * event_study['std'] / np.sqrt(event_study['count'])
    event_study['ci_upper'] = event_study['mean'] + 1.96 * event_study['std'] / np.sqrt(event_study['count'])
    
    # Plot
    plt.figure(figsize=(10, 6))
    plt.axhline(y=0, color='black', linestyle='--', alpha=0.5)
    plt.axvline(x=0, color='red', linestyle='--', alpha=0.5, label='Treatment')
    plt.plot(event_study.index, event_study['mean'], 'o-', color='blue', markersize=8)
    plt.fill_between(event_study.index, event_study['ci_lower'], event_study['ci_upper'], 
                     alpha=0.2, color='blue')
    plt.xlabel('Time Relative to Treatment')
    plt.ylabel('ATT(g,t)')
    plt.title('Event Study: Dynamic Treatment Effects')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()
    
    return event_study

## 4. Run Analysis

In [ ]:
# Compute ATT(g,t)
att_results = compute_att_gt(df)

print("ATT(g,t) Results:")
print(att_results.head(10))

# Plot event study
event_study = event_study_plot(att_results)

print("\nAverage Effect by Event Time:")
print(event_study[['mean', 'ci_lower', 'ci_upper']].round(2))

## 5. Two-Way Fixed Effects (TWFE) Comparison

In [ ]:
# Traditional TWFE regression
import statsmodels.formula.api as smf

# Create state and year fixed effects
df['post'] = (df['year'] >= df['treat_time']).astype(int)
df.loc[df["treat_time"].isna(), "post"] = 0

# TWFE regression
twfe = smf.ols('employment ~ treated + C(state) + C(year)', data=df).fit()

print("Traditional TWFE Results:")
print(f"Coefficient on treated: {twfe.params["treated"]:.3f}")
print(f"Standard error: {twfe.bse["treated"]:.3f}")
print(f"t-statistic: {twfe.tvalues["treated"]:.3f}")

print("\n" + "="*60)
print("Key Finding:")
print("The event study shows dynamic treatment effects over time.")
print("Pre-treatment coefficients should be close to zero (parallel trends).")
print("Post-treatment coefficients show the causal effect.")
print("="*60)

## Summary

This notebook implements staggered DiD following Callaway & Sant'Anna (2021):

1. **ATT(g,t)**: Group-time average treatment effects
2. **Event Study**: Dynamic effects over time
3. **Parallel Trends**: Check pre-treatment coefficients
4. **TWFE Comparison**: Compare with traditional approach

**Extensions to try**:
- Compare TWFE vs Callaway-Sant'Anna vs Sun-Abraham vs Borusyak
- Test sensitivity to donor pool composition
- Implement Doubly Robust DiD
- Use real Card-Krueger or QCEW data